In [1]:
import numpy as np
import pandas as pd

train = np.loadtxt("../ECG5000/ECG5000_TRAIN.txt")
test  = np.loadtxt("../ECG5000/ECG5000_TEST.txt")

y_train, X_train = train[:, 0].astype(int), train[:, 1:]
y_test,  X_test  = test[:, 0].astype(int),  test[:, 1:]

X_train.shape

(500, 140)

In [2]:
annotations = {
    1: "Normal",
    2: "R-on-T Premature Ventricular Contraction",
    3: "Premature Ventricular Contraction",
    4: "Supraventricular Premature or Ectopic Beat",
    5: "Unclassified Beat"
}

#fuse the training and test data into a single dataframe for easier analysis
df_train = pd.DataFrame(X_train)
df_test = pd.DataFrame(X_test)
df = pd.concat([df_train, df_test], axis=0)
df['Class'] = np.concatenate([y_train, y_test], axis=0)

In [3]:
for class_label, class_name in annotations.items():
    print(f"Class {class_label}: {class_name}, Count: {np.sum(df['Class'] == class_label)}")

Class 1: Normal, Count: 2919
Class 2: R-on-T Premature Ventricular Contraction, Count: 1767
Class 3: Premature Ventricular Contraction, Count: 96
Class 4: Supraventricular Premature or Ectopic Beat, Count: 194
Class 5: Unclassified Beat, Count: 24


In [4]:
df.T.describe()

,0,1,2,3,4,5,6,7,8,9,...,4490,4491,4492,4493,4494,4495,4496,4497,4498,4499
count,141.000000,141.000000,141.000000,141.000000,141.000000,141.000000,141.000000,141.000000,141.000000,141.000000,...,141.000000,141.000000,141.000000,141.000000,141.000000,141.000000,141.000000,141.000000,141.000000,141.000000
mean,0.007092,0.007092,0.007092,0.007092,0.007092,0.007092,0.007092,0.007092,0.007092,0.007092,...,0.014184,0.028369,0.014184,0.014184,0.014184,0.028369,0.014184,0.014184,0.014184,0.014184
std,0.999975,0.999975,0.999975,0.999975,0.999975,0.999975,0.999975,0.999975,0.999975,0.999975,...,1.010557,1.051823,1.010557,1.010557,1.010557,1.051823,1.010557,1.010557,1.010557,1.010557
min,-4.376041,-4.506579,-4.584095,-4.318823,-4.338224,-4.478011,-4.589669,-4.604201,-4.704657,-2.853671,...,-3.796142,-3.472208,-4.019493,-2.957065,-3.748719,-4.176790,-3.966026,-3.627311,-3.634981,-3.417164
25%,-0.367913,-0.249270,-0.434112,-0.391789,-0.491351,-0.377785,-0.394218,-0.351980,-0.287147,-0.630124,...,0.056170,-0.379497,-0.074151,-0.610599,-0.393668,-0.021798,-0.335928,0.079122,0.118497,-0.463884
50%,0.123925,0.185186,0.138290,0.074395,0.109887,-0.052632,0.125991,0.033968,0.120415,-0.176786,...,0.429674,-0.097026,0.435440,-0.142858,-0.133057,0.307696,0.040911,0.293628,0.395451,-0.102570
75%,0.481512,0.379831,0.498433,0.521975,0.558965,0.478305,0.457562,0.598844,0.554714,0.680506,...,0.572001,0.367296,0.570057,0.545838,0.595520,0.597168,0.632990,0.608974,0.549989,0.624451
max,2.125341,2.164346,2.262985,2.000769,1.741686,2.042437,1.948500,1.882810,2.393528,2.227544,...,2.000000,4.000000,2.000000,2.704595,2.433375,4.000000,2.280888,2.000000,2.000000,2.569073


In [ ]:
from statistics import NormalDist

def confidence_interval(data, confidence=0.95):
  # data: 2D array of shape (n_samples, n_timesteps)
  # returns the +/- half-width of the CI for the mean at each timestep
  data = np.asarray(data)
  n = data.shape[0]
  std = data.std(axis=0, ddof=1)
  z = NormalDist().inv_cdf((1 + confidence) / 2.)
  h = z * std / (n ** .5)
  return h


In [ ]:
import plotly.express as px

for class_label, class_name in annotations.items():
    #show the average waveform for each class with 95% CI
    class_data = df[df['Class'] == class_label].iloc[:, :-1]
    fig = px.line(
        class_data.mean(),
        title=f"Average waveform for class {class_label}: {class_name}",
        error_y=dict(
            type='data',
            array=confidence_interval(class_data.values),
            visible=True
        )
    )
    fig.show()
